# 4D-STEM NBD Grain Segmentation and {111}c Planar-Defect Classification — Part 1

**Pipeline:** preprocessing → NMF decomposition → grain segmentation → adjacency/peak-similarity merging → cleaned grain-boundary map.

This notebook processes a single 4D-STEM nanobeam-diffraction (NBD) dataset (one ROI of one composition) and saves an intermediate workspace state (`<dataset>_state.pkl`) that is consumed by **Part 2** for virtual-image visualization, defect-grain selection, and statistics.

To process a different ROI, change `dataset_name` and the input path in the configuration cell, then re-run.

> Raw `.h5` 4D-STEM datasets are not included in this repository; download them from the archive listed in the Data Availability Statement of the paper and place them under `./data/`.

In [ ]:
import os

# Set the GPU to use (e.g. "0") only when not already configured by the environment.
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import h5py
import numpy as np
from os.path import splitext
import py4DSTEM
import tifffile
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from matplotlib.colors import Normalize

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
# Change `dataset_name` and `filepath_data` to process a different ROI / composition.
# Expected input: a 4D-STEM NBD dataset in .h5 format (see Data Availability Statement).
dataset_name = "Gua20_Acq1"
filepath_data = f"./data/{dataset_name}.h5"          # place the raw .h5 here
filepath_save = splitext(filepath_data)[0] + "_preprocessed_filtered.h5"
save_prefix   = f"./results/{dataset_name}_"

os.makedirs("./results", exist_ok=True)

# Real- and reciprocal-space binning factors (1 = no binning; ROI export uses RAW data).
R_BIN = 1   # real-space (scan) binning
Q_BIN = 1   # reciprocal-space (diffraction) binning

# Load and preprocess the datacube.
datacube = py4DSTEM.read(filepath_data)
datacube.bin_R(R_BIN)
datacube.bin_Q(Q_BIN)
py4DSTEM.save(filepath_save, datacube, mode="o")   # save preprocessed copy (for ROI export)

In [ ]:
# --------------------------------------------------
# [Preprocessing - tuning] Central (direct-beam) mask setup and visualization
# --------------------------------------------------
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# 1) Mask center and radius. Adjust these values while inspecting the plot below.
mask_center_y = 64.5  # diffraction-pattern center, Y (pixels)
mask_center_x = 63    # diffraction-pattern center, X (pixels)
mask_radius   = 8.0   # mask radius (pixels)

# 2) Representative diffraction pattern (mean over all scan positions).
raw_dp = np.mean(datacube.data, axis=(0, 1))
Q_Ny, Q_Nx = raw_dp.shape

# 3) Build the circular (direct-beam) mask.
y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
dist_from_center = np.sqrt((y_grid - mask_center_y)**2 + (x_grid - mask_center_x)**2)
circular_mask = dist_from_center <= mask_radius

# 4) Apply the test mask.
masked_test_dp = raw_dp.copy()
masked_test_dp[circular_mask] = 0

# 5) Visualize the mask boundary and the masked result.
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(raw_dp**0.3, cmap="viridis")
circle = plt.Circle((mask_center_x, mask_center_y), mask_radius, color="cyan",
                    fill=False, linewidth=2, linestyle="--")
axes[0].add_patch(circle)
axes[0].set_title("Original Diffraction Pattern (with Mask Boundary)")
axes[1].imshow(masked_test_dp**0.3, cmap="viridis")
axes[1].set_title("Masked Result")
for ax in axes:
    ax.axis("on")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# --------------------------------------------------
# [Preprocessing - apply] Apply the finalized mask to the full dataset and binarize
# --------------------------------------------------
modified_data = np.array(datacube.data[:, :], dtype=np.float32)

# Rebuild the circular mask and apply it across the whole dataset.
Q_Ny, Q_Nx = modified_data.shape[2], modified_data.shape[3]
y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
dist_from_center = np.sqrt((y_grid - mask_center_y)**2 + (x_grid - mask_center_x)**2)
circular_mask = dist_from_center <= mask_radius
modified_data[:, :, circular_mask] = 0

# Binary thresholding: keep only peak positions as binary features for NMF.
binary_threshold = 5500
modified_data = np.where(modified_data > binary_threshold, 1, 0)
modified_data = np.nan_to_num(modified_data, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Circular mask (center: {mask_center_y}, {mask_center_x}, radius: {mask_radius}) applied.")
print(f"Binary threshold ({binary_threshold}) applied. Processed data shape: {modified_data.shape}")

In [ ]:
# --------------------------------------------------
# NMF decomposition (GPU, multiplicative updates)
# --------------------------------------------------
# Each grain in a near-parallel NBD scan produces a roughly constant diffraction pattern,
# so NMF on the binarized peak-position data factors the scan into grain-like components.
#
# NOTE on n_components:
#   Set this to roughly the number of distinct grains expected in the ROI (a slight
#   overestimate is fine; over-segmented fragments are merged in the next steps).
#   ~50 is a reasonable starting value for these ~128x128 scans; increase it for
#   ROIs with many small grains and decrease it for ROIs with few large grains.
n_components = 50

import torch

def fit_transform_nmf_gpu(X, n_components=50, max_iter=2000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
    M, N = X_tensor.shape
    K = n_components

    # Positive random initialization.
    W_tensor = torch.rand(M, K, dtype=torch.float32, device=device) + 0.1
    H_tensor = torch.rand(K, N, dtype=torch.float32, device=device) + 0.1
    eps = 1e-9

    for i in range(max_iter):
        # Update H.
        Wt = W_tensor.t()
        H_tensor *= (Wt @ X_tensor) / ((Wt @ W_tensor) @ H_tensor + eps)
        # Update W.
        Ht = H_tensor.t()
        W_tensor *= (X_tensor @ Ht) / (W_tensor @ (H_tensor @ Ht) + eps)

        if (i + 1) % 100 == 0 or i == 0:
            with torch.no_grad():
                rec_err = torch.norm(X_tensor - W_tensor @ H_tensor).item()
            print(f"Iteration {i+1}/{max_iter} - Reconstruction Error: {rec_err:.4f}")

    return W_tensor.cpu().numpy(), H_tensor.cpu().numpy()

# Flatten diffraction space and run NMF.
flattened_data = modified_data.reshape((-1, modified_data.shape[2] * modified_data.shape[3]))
X_np = np.nan_to_num(flattened_data, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
W, H = fit_transform_nmf_gpu(X_np, n_components=n_components, max_iter=2000)

In [ ]:
# --------------------------------------------------
# [Visualization] Combined RGB overlay of NMF components
# --------------------------------------------------
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

cmap_hsv = plt.colormaps["hsv"].resampled(n_components)
colors = [cmap_hsv(i)[:3] for i in range(n_components)]

combined_image = np.zeros((modified_data.shape[0], modified_data.shape[1], 3))
for i in range(n_components):
    feature_map = W[:, i].reshape(modified_data.shape[0], modified_data.shape[1])
    for j in range(3):
        combined_image[:, :, j] += feature_map * colors[i][j]

if combined_image.max() > 0:
    combined_image = combined_image / combined_image.max()
combined_image = np.clip(combined_image, 0, 1)

fig_comb = plt.figure(figsize=(10, 10))
plt.imshow(combined_image)
plt.axis("off")
plt.show()
plt.close(fig_comb)

In [ ]:
# ==================================================
# Island decomposition: tuning & preview (single component)
# ==================================================
import scipy.ndimage as ndimage
import matplotlib.pyplot as plt
import numpy as np

comp_idx         = 1     # NMF component index to preview (0 .. n_components-1)
tuning_threshold = 0.25  # normalized intensity threshold on the feature map (0..1)
min_island_area  = 10    # minimum island size (pixels) for noise filtering
select_island_idx = 0    # island index to preview the mean diffraction pattern for
exponent         = 0.3   # power-scale exponent for diffraction-pattern display

raw_crop = np.array(datacube.data[:, :], dtype=np.float32)

feature_img = W[:, comp_idx].reshape(modified_data.shape[0], modified_data.shape[1])
normalized_feat = (feature_img - feature_img.min()) / (feature_img.max() - feature_img.min() + 1e-8)

binary_mask = normalized_feat >= tuning_threshold
labeled_array, num_features = ndimage.label(binary_mask)

valid_islands = []
for j in range(1, num_features + 1):
    if np.sum(labeled_array == j) >= min_island_area:
        valid_islands.append(labeled_array == j)

num_islands = len(valid_islands)
print(f"Component {comp_idx}: {num_islands} valid islands (min size filter: {min_island_area}px)")

fig_tune, axes_tune = plt.subplots(1, 3, figsize=(18, 5))
axes_tune[0].imshow(normalized_feat, cmap="inferno")
axes_tune[0].set_title("Normalized feature map")
axes_tune[1].imshow(binary_mask, cmap="gray")
axes_tune[1].set_title(f"Binary mask (>= {tuning_threshold})")
axes_tune[2].imshow(labeled_array, cmap="nipy_spectral")
axes_tune[2].set_title(f"Labeled islands ({num_features})")
for ax in axes_tune:
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig_tune)

In [ ]:
# ==================================================
# Island decomposition & grain extraction (all components)
# ==================================================
import scipy.ndimage as ndimage
import numpy as np

global_threshold = globals().get("tuning_threshold", 0.3)  # feature-map threshold (0..1)
edge_threshold   = 0.05                                     # lower bound to avoid losing grain edges
min_island_area  = globals().get("min_island_area", 10)    # min island size (pixels)
exponent         = globals().get("exponent", 0.3)
dp_vmax_percentile  = 99.5
dp_brightness_factor = 1.0
colormap_name    = "inferno"

raw_crop = np.array(datacube.data[:, :], dtype=np.float32)

# Decompose each NMF component into individual grains (islands).
all_grains = []
for idx in range(n_components):
    feature_img = W[:, idx].reshape(modified_data.shape[0], modified_data.shape[1])
    normalized_feat = (feature_img - feature_img.min()) / (feature_img.max() - feature_img.min() + 1e-8)

    # Core seeds.
    binary_mask = normalized_feat >= global_threshold
    labeled_array, num_features = ndimage.label(binary_mask)

    # Label propagation to recover grain edges (avoid boundary loss).
    if num_features > 0:
        edge_mask = normalized_feat >= edge_threshold
        distances, indices = ndimage.distance_transform_edt(
            ~binary_mask, return_distances=True, return_indices=True)
        propagated = labeled_array[tuple(indices)]
        propagated[~edge_mask] = 0

        for lab in range(1, num_features + 1):
            grain_mask = propagated == lab
            area = int(np.sum(grain_mask))
            if area >= min_island_area:
                feat = np.zeros_like(normalized_feat)
                feat[grain_mask] = normalized_feat[grain_mask]
                ys, xs = np.where(grain_mask)
                all_grains.append({
                    "component": idx,
                    "mask": grain_mask,
                    "feature": feat,
                    "area": area,
                    "centroid": (float(ys.mean()), float(xs.mean())),
                })

print(f"Initial grain count (before merging): {len(all_grains)}")

In [ ]:
# ==============================================================================
# Adjacency & reciprocal-space peak-similarity based merge proposals
# ==============================================================================
import scipy.ndimage as ndimage
import numpy as np

peak_similarity_threshold = 0.81  # diffraction-peak overlap threshold (0..1)
max_centroid_distance     = 4.0   # max real-space centroid distance for a merge (pixels)
bin_thresh = binary_threshold
c_mask = globals().get("circular_mask", None)

if c_mask is None:
    Q_Ny, Q_Nx = raw_crop.shape[2], raw_crop.shape[3]
    y_grid, x_grid = np.ogrid[:Q_Ny, :Q_Nx]
    c_mask = np.sqrt((y_grid - globals().get("mask_center_y", 64.5))**2 +
                     (x_grid - globals().get("mask_center_x", 63.5))**2) <= globals().get("mask_radius", 8.0)

num_initial_grains = len(all_grains)
print(f"[Merge detection] grains before merging: {num_initial_grains}")

# 1) Binary diffraction-peak profile per grain.
grain_peaks = []
for idx in range(num_initial_grains):
    g = all_grains[idx]
    avg_raw_dp = np.mean(raw_crop[g["mask"]], axis=0)
    peaks = (avg_raw_dp > bin_thresh) & (~c_mask)
    grain_peaks.append(peaks)

# 2) Build adjacency graph: real-space neighbors with similar reciprocal-space peaks.
adj_graph = {i: set() for i in range(num_initial_grains)}
for i in range(num_initial_grains):
    mask_i  = all_grains[i]["mask"]
    peaks_i = grain_peaks[i]
    n_i = np.sum(peaks_i)
    dilated_i = ndimage.binary_dilation(mask_i)
    cy_i, cx_i = all_grains[i]["centroid"]
    for j in range(i + 1, num_initial_grains):
        if not np.any(dilated_i & all_grains[j]["mask"]):
            continue
        cy_j, cx_j = all_grains[j]["centroid"]
        if np.hypot(cy_i - cy_j, cx_i - cx_j) > max_centroid_distance:
            continue
        peaks_j = grain_peaks[j]
        n_j = np.sum(peaks_j)
        inter = np.sum(peaks_i & peaks_j)
        denom = min(n_i, n_j) if min(n_i, n_j) > 0 else 1
        if inter / denom >= peak_similarity_threshold:
            adj_graph[i].add(j)
            adj_graph[j].add(i)

# 3) Connected components -> merge groups.
visited, detected_merge_groups = set(), []
for node in range(num_initial_grains):
    if node in visited:
        continue
    queue, comp = [node], []
    visited.add(node)
    while queue:
        curr = queue.pop(0)
        comp.append(curr)
        for nb in adj_graph[curr]:
            if nb not in visited:
                visited.add(nb)
                queue.append(nb)
    if len(comp) > 1:
        detected_merge_groups.append(sorted(comp))

initial_all_grains = list(all_grains)
print(f"Proposed merge groups: {len(detected_merge_groups)}")

In [ ]:
# ==============================================================================
# Apply merges & rebuild the grain list
# ==============================================================================
import numpy as np

initial_grains  = globals().get("initial_all_grains", all_grains)
proposed_groups = globals().get("detected_merge_groups", [])
# Accept all proposed merges by default; edit this list to accept a subset.
accepted_indices = globals().get("accepted_merge_indices", list(range(len(proposed_groups))))
print(f"Accepted merge-group indices: {accepted_indices}")

num_init = len(initial_grains)
adj_graph_final = {i: set() for i in range(num_init)}
for idx in accepted_indices:
    if idx < len(proposed_groups):
        gp = proposed_groups[idx]
        for i in range(len(gp)):
            for j in range(i + 1, len(gp)):
                adj_graph_final[gp[i]].add(gp[j])
                adj_graph_final[gp[j]].add(gp[i])

visited_final, final_groups = set(), []
for node in range(num_init):
    if node in visited_final:
        continue
    queue, comp = [node], []
    visited_final.add(node)
    while queue:
        curr = queue.pop(0)
        comp.append(curr)
        for nb in adj_graph_final[curr]:
            if nb not in visited_final:
                visited_final.add(nb)
                queue.append(nb)
    final_groups.append(sorted(comp))

# Merge grain masks/features within each final group.
merged_grains = []
for group in final_groups:
    mask = np.zeros_like(initial_grains[0]["mask"], dtype=bool)
    feat = np.zeros_like(initial_grains[0]["feature"], dtype=float)
    for idx in group:
        mask |= initial_grains[idx]["mask"]
        feat = np.maximum(feat, initial_grains[idx]["feature"])
    ys, xs = np.where(mask)
    merged_grains.append({
        "members": group,
        "mask": mask,
        "feature": feat,
        "area": int(np.sum(mask)),
        "centroid": (float(ys.mean()), float(xs.mean())),
    })

all_grains = merged_grains
num_grains = len(all_grains)
print(f"Grain count after merging: {num_grains}")

In [ ]:
# ==================================================
# Grain combined RGB overlay (no DP, black background)
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os

if num_grains > 0:
    H, W_grid = modified_data.shape[0], modified_data.shape[1]
    cmap_hsv = plt.colormaps["hsv"].resampled(num_grains)
    grain_colors = [cmap_hsv(i)[:3] for i in range(num_grains)]
    np.random.seed(123)
    np.random.shuffle(grain_colors)

    combined_image = np.zeros((H, W_grid, 3))
    for i in range(num_grains):
        grain_feat = all_grains[i]["feature"]
        for j in range(3):
            combined_image[:, :, j] += grain_feat * grain_colors[i][j]
    if combined_image.max() > 0:
        combined_image = combined_image / combined_image.max()
    combined_image = np.clip(combined_image, 0, 1)

    fig_comb = plt.figure(figsize=(10, 10))
    plt.imshow(combined_image)
    plt.axis("off")
    plt.savefig(save_prefix + "_grain_combined_overlay.png", bbox_inches="tight", dpi=150)
    print("Saved:", save_prefix + "_grain_combined_overlay.png")
    plt.show()
    plt.close(fig_comb)

In [ ]:
# ==================================================
# Grain overlay with pixel-boundary lines
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np

if num_grains > 0:
    H, W_grid = modified_data.shape[0], modified_data.shape[1]

    # Dominant grain label per pixel (background uses edge_threshold).
    edge_th = globals().get("edge_threshold", 0.05)
    features_3d = np.zeros((num_grains + 1, H, W_grid))
    features_3d[0, :, :] = edge_th
    for idx, g in enumerate(all_grains):
        features_3d[idx + 1] = g["feature"]
    L = np.argmax(features_3d, axis=0)

    # Build pixel-boundary segments.
    segments = []
    for y in range(H - 1):
        for x in range(W_grid):
            if L[y, x] != L[y + 1, x]:
                segments.append([(x - 0.5, y + 0.5), (x + 0.5, y + 0.5)])
    for y in range(H):
        for x in range(W_grid - 1):
            if L[y, x] != L[y, x + 1]:
                segments.append([(x + 0.5, y - 0.5), (x + 0.5, y + 0.5)])

    fig_comb = plt.figure(figsize=(10, 10))
    ax = fig_comb.add_subplot(111)
    ax.imshow(np.zeros((H, W_grid)), cmap="gray", vmin=0, vmax=1)
    ax.add_collection(LineCollection(segments, colors="white", linewidths=1.0))
    ax.axis("off")
    plt.show()
    plt.close(fig_comb)

In [ ]:
# ==================================================
# Clean small closed regions (<= 4 px) and finalize label map L_clean
# ==================================================
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import scipy.ndimage as ndimage
import numpy as np

if num_grains > 0:
    H, W_grid = modified_data.shape[0], modified_data.shape[1]
    L_clean = L.copy()

    # Iteratively merge connected components of size <= 4 px into the dominant neighbor.
    while True:
        changed = False
        for val in np.unique(L_clean):
            labeled_components, num_components = ndimage.label(L_clean == val)
            for c in range(1, num_components + 1):
                comp_mask = labeled_components == c
                if np.sum(comp_mask) <= 4:
                    dilated = ndimage.binary_dilation(comp_mask)
                    border_mask = dilated & ~comp_mask
                    neighbor_labels = L_clean[border_mask]
                    neighbor_labels = neighbor_labels[neighbor_labels != val]
                    if neighbor_labels.size > 0:
                        vals, counts = np.unique(neighbor_labels, return_counts=True)
                        L_clean[comp_mask] = vals[np.argmax(counts)]
                        changed = True
        if not changed:
            break

    # Cleaned grain-boundary segments.
    segments_clean = []
    for y in range(H - 1):
        for x in range(W_grid):
            if L_clean[y, x] != L_clean[y + 1, x]:
                segments_clean.append([(x - 0.5, y + 0.5), (x + 0.5, y + 0.5)])
    for y in range(H):
        for x in range(W_grid - 1):
            if L_clean[y, x] != L_clean[y, x + 1]:
                segments_clean.append([(x + 0.5, y - 0.5), (x + 0.5, y + 0.5)])

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)
    ax.imshow(np.zeros((H, W_grid)), cmap="gray", vmin=0, vmax=1)
    ax.add_collection(LineCollection(segments_clean, colors="white", linewidths=1.0))
    ax.axis("off")
    plt.show()
    plt.close(fig)

In [ ]:
# ==============================================================================
# Save workspace state for Part 2
# ==============================================================================
import pickle
import os

state = {
    "dataset_name": dataset_name,
    "save_prefix": save_prefix,
    "filepath_data": filepath_data,
    "filepath_save": filepath_save,
    "all_grains": all_grains,
    "num_grains": num_grains,
    "modified_data": modified_data,
    "L": L,
    "L_clean": L_clean,
    "raw_crop": raw_crop,
    "c_mask": c_mask,
    "circular_mask": circular_mask if "circular_mask" in globals() else None,
    "mask_center_y": mask_center_y,
    "mask_center_x": mask_center_x,
    "mask_radius": mask_radius,
    "exponent": exponent,
    "dp_vmax_percentile": dp_vmax_percentile,
    "dp_brightness_factor": dp_brightness_factor,
    "colormap_name": colormap_name,
}
os.makedirs("./results", exist_ok=True)
state_path = f"./results/{dataset_name}_state.pkl"
with open(state_path, "wb") as f:
    pickle.dump(state, f)
print(f"Workspace state saved to: {state_path}")